In [1]:
import pandas as pd

df1 = pd.read_csv("../data/raw/purplle_campaign_data_with_nulls.csv")
df2 = pd.read_csv("../data/raw/nykaa_campaign_data_with_nulls.csv")
df3 = pd.read_csv("../data/raw/tira_campaign_data_with_nulls.csv")

# add a source column (optional but useful)
df1["Brand"] = "Tira"
df2["Brand"] = "Purplle"
df3["Brand"] = "Nykaa"

# combine vertically
df = pd.concat([df1, df2, df3], ignore_index=True)

In [2]:
print('row , column')
print(df.shape)

print('\ncolumn names')
print(df.columns)

print('\ntypes of columns')
print(df.dtypes)

print('\n information about data frame')
print(df.info())

row , column
(166665, 17)

column names
Index(['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration',
       'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions',
       'Revenue', 'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score',
       'Customer_Segment', 'Date', 'Brand'],
      dtype='str')

types of columns
Campaign_ID             str
Campaign_Type           str
Target_Audience         str
Duration            float64
Channel_Used            str
Impressions         float64
Clicks              float64
Leads               float64
Conversions         float64
Revenue             float64
Acquisition_Cost    float64
ROI                 float64
Language                str
Engagement_Score    float64
Customer_Segment        str
Date                    str
Brand                   str
dtype: object

 information about data frame
<class 'pandas.DataFrame'>
RangeIndex: 166665 entries, 0 to 166664
Data columns (total 17 columns):
 #   Column            Non-Null Count 

In [3]:
df.head()

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date,Brand
0,PU-CMP-1000,Paid Ads,Working Women,6.0,"Facebook, Google",31869.0,3490.0,1946.0,902.0,493394.0,55.93,8.78,Tamil,19.89,Premium Shoppers,27-12-2024,Tira
1,PU-CMP-1001,Paid Ads,Premium Shoppers,8.0,"Instagram, WhatsApp",26640.0,2441.0,741.0,314.0,108644.0,197.44,0.75,Bengali,13.12,NaN,26-11-2024,Tira
2,PU-CMP-1002,Influencer,Youth,22.0,"Facebook, WhatsApp",36967.0,4997.0,1025.0,653.0,NaN,234.59,0.56,NaN,18.06,College Students,08-01-2025,Tira
3,PU-CMP-1003,Paid Ads,Youth,15.0,"Instagram, WhatsApp",73258.0,5893.0,2565.0,1970.0,1341570.0,103.76,5.56,Bengali,14.23,College Students,14-03-2025,Tira
4,PU-CMP-1004,Social Media,Premium Shoppers,11.0,"WhatsApp, Instagram",91081.0,6393.0,3282.0,2582.0,787510.0,78.61,NaN,English,13.46,College Students,11-07-2024,Tira


In [4]:
print("null count for each column")
print(df.isnull().sum())
print("\ntotal count of row which has null value")
print(df.isnull().any(axis=1).sum())
print("\ntotal count of row which has duplicate value")
print(df.duplicated().sum())


null count for each column
Campaign_ID         5417
Campaign_Type       8358
Target_Audience     8172
Duration            8323
Channel_Used        8282
Impressions         8224
Clicks              8224
Leads               8090
Conversions         8000
Revenue             8304
Acquisition_Cost    8161
ROI                 8298
Language            8299
Engagement_Score    8081
Customer_Segment    8175
Date                8099
Brand                  0
dtype: int64

total count of row which has null value
91104

total count of row which has duplicate value
0


In [5]:
df.drop(columns=['Campaign_ID' ,'Language' ,'Date'], inplace=True)

string_cols = df.select_dtypes(include='str').columns

df[string_cols] = df[string_cols].apply(
    lambda x: x.str.strip()          # remove leading/trailing spaces
               .str.lower()           # convert to lowercase
               .str.replace('  ', ' ')  # remove double spaces
)

In [6]:
map_ch = df.groupby("Campaign_Type")["Channel_Used"].agg(lambda x: x.mode()[0])

df["Channel_Used"] = df["Channel_Used"].fillna(
    df["Campaign_Type"].map(map_ch)
)

In [7]:
map_ct = df.groupby("Channel_Used")["Campaign_Type"].agg(lambda x: x.mode()[0])

df["Campaign_Type"] = df["Campaign_Type"].fillna(
    df["Channel_Used"].map(map_ct)
)

In [8]:
# Broad → Niche: for each Customer_Segment, what Target_Audience commonly pairs with it?
mapping_to_niche = df.dropna(subset=["Customer_Segment", "Target_Audience"]) \
                     .drop_duplicates(subset=["Customer_Segment"]) \
                     .set_index("Customer_Segment")["Target_Audience"]


# Fill missing Target_Audience using Customer_Segment mapping
df["Target_Audience"] = df["Target_Audience"].fillna(df["Customer_Segment"].map(mapping_to_niche))

In [9]:
# Niche → Broad: for each Target_Audience, what Customer_Segment commonly pairs with it?
mapping_to_broad = df.dropna(subset=["Customer_Segment", "Target_Audience"]) \
                     .drop_duplicates(subset=["Target_Audience"]) \
                     .set_index("Target_Audience")["Customer_Segment"]

# Fill missing Customer_Segment using Target_Audience mapping
df["Customer_Segment"] = df["Customer_Segment"].fillna(df["Target_Audience"].map(mapping_to_broad))

In [10]:
critical_cols = ["Campaign_Type", "Channel_Used", "Acquisition_Cost", "Target_Audience", "Customer_Segment", "Impressions"]

for col in critical_cols:
    before = len(df)
    df = df.dropna(subset=[col])
    after = len(df)
    print(f"{col}: dropped {before - after} rows, {after} rows remaining")

Campaign_Type: dropped 411 rows, 166254 rows remaining
Channel_Used: dropped 0 rows, 166254 rows remaining
Acquisition_Cost: dropped 8132 rows, 158122 rows remaining
Target_Audience: dropped 395 rows, 157727 rows remaining
Customer_Segment: dropped 0 rows, 157727 rows remaining
Impressions: dropped 7788 rows, 149939 rows remaining


In [11]:
group_cols = ["Campaign_Type", "Channel_Used", "Target_Audience"]

fill clicks null values using ( clicks / impressions = CTR ) -> ( impressions x CTR = click )

In [12]:
# 1. use all valid rows (no split)
df_temp = df[df["Clicks"].notna() & df["Impressions"].notna()].copy()

# 2. compute CTR
df_temp["CTR"] = df_temp["Clicks"] / df_temp["Impressions"]

# 3. IQR-based clipping (data-driven)
q1 = df_temp["CTR"].quantile(0.25)
q3 = df_temp["CTR"].quantile(0.75)
iqr = q3 - q1

lower = max(0, q1 - 1.5 * iqr)  # CTR can't be negative
upper = q3 + 1.5 * iqr

df_temp["CTR"] = df_temp["CTR"].clip(lower, upper)

# 4. group median CTR
ctr_group = df_temp.groupby(group_cols)["CTR"].median().reset_index()

# 5. merge back
df = df.merge(ctr_group, on=group_cols, how="left")


# 6. fallback (global median)
overall_ctr = df_temp["CTR"].median()
df["CTR"] = df["CTR"].fillna(overall_ctr)

# 7. fill missing Clicks
mask = df["Clicks"].isnull()
df.loc[mask, "Clicks"] = (
    df.loc[mask, "Impressions"] * df.loc[mask, "CTR"]
)

df["Clicks"] = df["Clicks"].round().astype(int)



fill lead null values with Lead Rate ( lead / clicks = lead rate ) -> ( clicks / lead = click rate )

In [13]:
# 1. use valid rows
df_temp = df[df["Leads"].notna() & df["Clicks"].notna()].copy()

# 2. compute Lead Rate
df_temp["Lead_Rate"] = df_temp["Leads"] / df_temp["Clicks"]

# 3. IQR-based clipping
q1 = df_temp["Lead_Rate"].quantile(0.25)
q3 = df_temp["Lead_Rate"].quantile(0.75)
iqr = q3 - q1

lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

df_temp["Lead_Rate"] = df_temp["Lead_Rate"].clip(lower, upper)

# 4. group median
lead_group = df_temp.groupby(group_cols)["Lead_Rate"].median().reset_index()

# 5. merge back
df = df.merge(lead_group, on=group_cols, how="left")

# 6. fallback
overall_lead = df_temp["Lead_Rate"].median()
df["Lead_Rate"] = df["Lead_Rate"].fillna(overall_lead)

# 7. fill Leads
mask = df["Leads"].isnull()
df.loc[mask, "Leads"] = (
    df.loc[mask, "Clicks"] * df.loc[mask, "Lead_Rate"]
)

df["Leads"] = df["Leads"].round().astype(int)


fill converstion null values using ( leads / conversions = conversion rate ) -> ( leads × conversion rate = conversions )

In [14]:
# 1. use valid rows
df_temp = df[df["Conversions"].notna() & df["Leads"].notna()].copy()

# 2. compute Conversion Rate
df_temp["Conversion_Rate"] = df_temp["Conversions"] / df_temp["Leads"]

# 3. IQR-based clipping
q1 = df_temp["Conversion_Rate"].quantile(0.25)
q3 = df_temp["Conversion_Rate"].quantile(0.75)
iqr = q3 - q1

lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

df_temp["Conversion_Rate"] = df_temp["Conversion_Rate"].clip(lower, upper)

# 4. group median
conv_group = df_temp.groupby(group_cols)["Conversion_Rate"].median().reset_index()

# 5. merge back
df = df.merge(conv_group, on=group_cols, how="left")

# 6. fallback
overall_conv = df_temp["Conversion_Rate"].median()
df["Conversion_Rate"] = df["Conversion_Rate"].fillna(overall_conv)

# 7. fill Conversions
mask = df["Conversions"].isnull()
df.loc[mask, "Conversions"] = (
    df.loc[mask, "Leads"] * df.loc[mask, "Conversion_Rate"]
)

df["Conversions"] = df["Conversions"].round().astype(int)


fill revenue null values using ( Revenue / conversions = Value ) -> ( Value × conversion = revenue )

In [15]:
# 1. use valid rows
df_temp = df[df["Revenue"].notna() & df["Conversions"].notna()].copy()

# 2. compute value per conversion
df_temp["Rev_per_Conv"] = df_temp["Revenue"] / df_temp["Conversions"]

# 3. remove extreme outliers (stronger than IQR → use quantile cap)
lower = df_temp["Rev_per_Conv"].quantile(0.05)
upper = df_temp["Rev_per_Conv"].quantile(0.95)

df_temp["Rev_per_Conv"] = df_temp["Rev_per_Conv"].clip(lower, upper)

# 4. group median (stable)
rev_group = df_temp.groupby(group_cols)["Rev_per_Conv"].median().reset_index()

# 5. merge back
df = df.merge(rev_group, on=group_cols, how="left")

# 6. fallback hierarchy (important improvement)
overall_rev = df_temp["Rev_per_Conv"].median()

df["Rev_per_Conv"] = df["Rev_per_Conv"].fillna(overall_rev)

# 7. fill Revenue
mask = df["Revenue"].isnull()
df.loc[mask, "Revenue"] = (
    df.loc[mask, "Conversions"] * df.loc[mask, "Rev_per_Conv"]
)

df["Revenue"] = df["Revenue"].round().astype(int)



In [16]:
group_cols = ["Campaign_Type", "Channel_Used"]

# 1. use valid rows
df_temp = df[df["Duration"].notna()].copy()

# 2. optional: IQR clipping (safe for duration)
q1 = df_temp["Duration"].quantile(0.25)
q3 = df_temp["Duration"].quantile(0.75)
iqr = q3 - q1

lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

df_temp["Duration"] = df_temp["Duration"].clip(lower, upper)

# 3. group median
dur_group = df_temp.groupby(group_cols)["Duration"].median().reset_index()

# 4. merge back
df = df.merge(dur_group, on=group_cols, how="left", suffixes=("", "_grp"))

# 5. fallback
overall_dur = df_temp["Duration"].median()
df["Duration"] = df["Duration"].fillna(df["Duration_grp"])
df["Duration"] = df["Duration"].fillna(overall_dur)

# 6. finalize
df["Duration"] = df["Duration"].round().astype(int)

# 7. cleanup
df.drop(columns=["Duration_grp"], inplace=True)



fill engagement null values using (clicks + leads / impressions) as engagement

In [17]:
group_cols = ["Campaign_Type", "Channel_Used", "Target_Audience"]

# 1. use valid rows
df_temp = df[
    df["Engagement_Score"].notna() &
    df["Clicks"].notna() &
    df["Leads"].notna() &
    df["Impressions"].notna()
].copy()

# 2. compute engagement rate
df_temp["Eng_rate"] = (df_temp["Clicks"] + df_temp["Leads"]) / df_temp["Impressions"]

# 3. IQR clipping
q1 = df_temp["Eng_rate"].quantile(0.25)
q3 = df_temp["Eng_rate"].quantile(0.75)
iqr = q3 - q1

lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

df_temp["Eng_rate"] = df_temp["Eng_rate"].clip(lower, upper)

# 4. map rate → score (learn scale from data)
# ratio between score and rate
df_temp["scale"] = df_temp["Engagement_Score"] / df_temp["Eng_rate"]

scale_group = df_temp.groupby(group_cols)["scale"].median().reset_index()

# 5. merge both
eng_group = df_temp.groupby(group_cols)["Eng_rate"].median().reset_index()

df = df.merge(eng_group, on=group_cols, how="left")
df = df.merge(scale_group, on=group_cols, how="left")

# 6. fallback
overall_rate = df_temp["Eng_rate"].median()
overall_scale = df_temp["scale"].median()

df["Eng_rate"] = df["Eng_rate"].fillna(overall_rate)
df["scale"] = df["scale"].fillna(overall_scale)

# 7. fill Engagement_Score
mask = df["Engagement_Score"].isnull()
df.loc[mask, "Engagement_Score"] = (
    df.loc[mask, "Eng_rate"] * df.loc[mask, "scale"]
)

df["Engagement_Score"] = df["Engagement_Score"].round().astype(int)

# 8. cleanup
df.drop(columns=["Eng_rate", "scale"], inplace=True)



In [18]:
# def winsorize(s):
#     return s.clip(s.quantile(0.01), s.quantile(0.99))

# cols = ["Clicks","Leads","Conversions","Revenue","Acquisition_Cost",]

# for col in cols:
#     df[col] = winsorize(df[col]).round()


In [19]:
# Recalculate ROI using standard formula (profit-based)

df["Total_Cost"] = df["Acquisition_Cost"] * df["Conversions"]

df["ROI"] = (
    (df["Revenue"] - df["Total_Cost"]) 
    / df["Total_Cost"]
).round(2)

In [20]:
df.to_csv('../data/cleaned/cleaned.csv', index=False)